# Can we predict next-month churn?
Target: `churn_next_month` (1 = account cancels in the month after the snapshot month).
Data: one row per account per month (account-month snapshots).

Plan: audit the data -> check for leakage -> build honest validation (grouped by account, and forward in time) -> compare to baselines -> interpret.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance

df = pd.read_csv('data.csv')
print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data audit

In [ ]:
print(df.shape, '| accounts:', df.account_ref.nunique())
print('nulls:', int(df.isna().sum().sum()), '| duplicate account-months:', int(df.duplicated(['account_ref','snapshot_month']).sum()))
print('churn rate: %.3f' % df.churn_next_month.mean())
print('rows per account:', df.groupby('account_ref').size().describe()[['min','50%','max']].to_dict())
df.describe().T

## 2. Leakage check: `lifetime_months`
The dictionary says `lifetime_months` is the *total months the account was active, from a CRM export of 2026-01-05* - i.e. it is measured **after** the snapshot, using knowledge of when the account ended. It would not be available at prediction time. Let's see how it relates to the target.

In [ ]:
df['lifetime_minus_tenure'] = df.lifetime_months - df.tenure_months
print(df.groupby('churn_next_month').lifetime_minus_tenure.describe())
print('\nchurn rate when lifetime == tenure:', df.loc[df.lifetime_minus_tenure==0,'churn_next_month'].mean())
print('churn rate when lifetime  > tenure:', df.loc[df.lifetime_minus_tenure>0,'churn_next_month'].mean())

Every churning row has `lifetime_months == tenure_months`, and no non-churning row does: the column **is** the label in disguise (the account's last active month is the snapshot month). Any model using it scores ~perfectly and would be useless in production. **It is excluded from all features below.** The next cells quantify the inflation.

## 3. Features and validation design

In [ ]:
df['month'] = pd.PeriodIndex(df.snapshot_month, freq='M')
df = df.sort_values(['account_ref','month']).reset_index(drop=True)

# Past-only history features (use only earlier snapshots of the same account)
g = df.groupby('account_ref')
df['usage_prev']   = g.usage_hours.shift(1)
df['usage_delta']  = df.usage_hours - df.usage_prev
df['tickets_prev'] = g.support_tickets.shift(1)
df['fee_per_seat'] = df.monthly_fee / df.seats
df['usage_per_seat'] = df.usage_hours / df.seats
df['month_num'] = df.month.dt.month   # NOT used in the main model - see section 3b

BASE = ['plan','seats','monthly_fee','tenure_months','usage_hours','support_tickets']
EXTRA = ['usage_prev','usage_delta','tickets_prev','fee_per_seat','usage_per_seat']
CAT = ['plan']
y = df.churn_next_month.values
groups = df.account_ref.values

## 3b. Churn is not stationary over time
Churn rate by snapshot month shows a one-off spike (snapshots 2024-11 to 2025-01), which did not recur a year later. A calendar-month feature can memorise that episode, which looks great in random/grouped CV but does not generalise forward. We check this explicitly.

In [ ]:
by_m = df.groupby('month').churn_next_month.agg(['mean','size'])
fig, ax = plt.subplots(figsize=(9,3.2))
ax.plot(by_m.index.astype(str), by_m['mean'], marker='o', color='#3b6ea5'); ax.set(ylabel='churn rate', title='Next-month churn rate by snapshot month')
plt.xticks(rotation=60); plt.tight_layout(); plt.show()
by_m.round(3).T

Validation matters here because accounts appear many times:
* **GroupKFold by account** - an account's rows never straddle train/test.
* **Forward-in-time split** - train on snapshots up to 2025-03, test on 2025-04 onward (closest to real deployment). Accounts can appear on both sides, as they would in deployment, but the model only ever sees the past.

In [ ]:
def make(model, feats):
    num = [f for f in feats if f not in CAT]
    pre = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), [f for f in feats if f in CAT]),
                             ('n', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()) if model=='lr' else 'passthrough', num)])
    est = (LogisticRegression(max_iter=1000) if model=='lr'
           else HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=0))
    return make_pipeline(pre, est)

def score(yt, p): return dict(roc_auc=roc_auc_score(yt,p), pr_auc=average_precision_score(yt,p))

def cv_eval(model, feats, n=5):
    oof = np.zeros(len(df))
    for tr, te in GroupKFold(n).split(df, y, groups):
        m = make(model, feats).fit(df.loc[tr, feats], y[tr])
        oof[te] = m.predict_proba(df.loc[te, feats])[:,1]
    return oof

def time_eval(model, feats, cutoff='2025-04'):
    tr = (df.month < pd.Period(cutoff)).values; te = ~tr
    m = make(model, feats).fit(df.loc[tr, feats], y[tr])
    return m, te, m.predict_proba(df.loc[te, feats])[:,1]

## 4. Results

In [ ]:
rows = []
rows.append(('Base rate (no skill)', 'grouped CV', dict(roc_auc=0.5, pr_auc=y.mean())))
# Leaky model - for illustration only
df['lifetime_months_LEAK'] = df.lifetime_months
leak = ['lifetime_months_LEAK'] + BASE
rows.append(('GBM + lifetime_months (LEAKY)', 'grouped CV', score(y, cv_eval('gbm', leak))))
# Honest models
for name, model, feats in [('Logistic, base features','lr',BASE), ('GBM, base features','gbm',BASE),
                           ('Logistic, + history features','lr',BASE+EXTRA), ('GBM, + history features','gbm',BASE+EXTRA)]:
    rows.append((name, 'grouped CV', score(y, cv_eval(model, feats))))
    _, te, p = time_eval(model, feats)
    rows.append((name, 'forward in time', score(y[te], p)))
# The calendar-month trap: same GBM plus month_num
rows.append(('GBM + history + month_num (TRAP)', 'grouped CV', score(y, cv_eval('gbm', BASE+EXTRA+['month_num']))))
_, te_, p_ = time_eval('gbm', BASE+EXTRA+['month_num'])
rows.append(('GBM + history + month_num (TRAP)', 'forward in time', score(y[te_], p_)))
res = pd.DataFrame([(a,b,v['roc_auc'],v['pr_auc']) for a,b,v in rows], columns=['model','validation','roc_auc','pr_auc'])
res.round(3)

The leaky row is what you would see if you trusted `lifetime_months` - near-perfect and meaningless. The honest rows are the real answer.

In [ ]:
# Look at the honest best model on the forward-in-time holdout
FEATS = BASE   # history features added no lift (section 4)
m, te, p = time_eval('gbm', FEATS)
yt = y[te]
order = np.argsort(-p)
print('Holdout rows: %d, churn rate %.3f' % (te.sum(), yt.mean()))
for frac in [0.05, 0.10, 0.20, 0.30]:
    k = int(frac*len(p)); top = order[:k]
    print('Top %2d%% by risk: churn rate %.3f  (lift %.2fx, captures %.1f%% of churners)'
          % (frac*100, yt[top].mean(), yt[top].mean()/yt.mean(), 100*yt[top].sum()/yt.sum()))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
bins = pd.qcut(p, 10, duplicates='drop')
ax[0].bar(range(len(bins.categories)), pd.Series(yt).groupby(bins).mean().values, color='#3b6ea5')
ax[0].axhline(yt.mean(), color='gray', ls='--', label='overall rate'); ax[0].legend()
ax[0].set(xlabel='predicted-risk decile (low -> high)', ylabel='observed churn rate', title='Calibration / ranking on holdout')
ax[1].hist(p[yt==0], bins=40, alpha=.6, density=True, label='stayed'); ax[1].hist(p[yt==1], bins=40, alpha=.6, density=True, label='churned')
ax[1].set(xlabel='predicted probability', title='Score distribution'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. What drives churn?

In [ ]:
Xte = df.loc[te, FEATS]
pi = permutation_importance(m, Xte, yt, scoring='average_precision', n_repeats=5, random_state=0)
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values()
imp.plot.barh(figsize=(6,4), color='#3b6ea5'); plt.xlabel('drop in PR-AUC when shuffled'); plt.tight_layout(); plt.show()

for c in ['tenure_months','usage_hours','support_tickets']:
    q = pd.qcut(df[c], 5, duplicates='drop')
    print(df.groupby(q).churn_next_month.mean().round(3).to_string(), '\n')

## 6. Conclusions
**Can we predict next-month churn? Only modestly.**

* **`lifetime_months` must not be used.** It is recorded in a later CRM export and equals `tenure_months` for every churning row and no other row, so it *is* the label. It gives AUC 1.00; that is leakage, not skill.
* **Honest performance** (base features: plan, seats, fee, tenure, usage, tickets): ROC-AUC about 0.67-0.68 and PR-AUC about 0.21 forward in time (base rate: 0.10 in the holdout), about 0.29 in grouped CV (base rate 0.15). That is real but weak signal. On the forward holdout the top 10% of accounts by risk churn at about 2.7x the base rate and contain about 27% of all churners. Useful for prioritising outreach, not for confident individual predictions.
* **Simple beats complex.** Logistic regression and gradient boosting perform about the same, and adding prior-month history (usage change, previous tickets) gives no lift.
* **Direction of effects** (section 5): low usage and more support tickets go with higher churn; plan type barely matters. These tables are raw associations and are partly confounded by the time effect below.
* **Calendar month is a trap.** Churn spiked to 40-55% for snapshots 2024-11 to 2025-01 and did not repeat in 2025-11. Adding `month_num` lifts grouped CV AUC to 0.80, but forward-in-time it is worse (0.63) - it memorises a one-off episode. Whatever caused that spike (pricing change? outage? data issue?) is worth finding out, because it is the biggest driver of churn in this data and the available features do not explain it.

**Caveats**
* Rows are correlated within account (about 3,000 accounts, 14,759 rows); we used grouped CV and a forward-in-time split. The forward holdout is a calmer period than training, so its AUC is probably optimistic or pessimistic in ways we can't quantify with one split.
* Previous-month features assume consecutive monthly snapshots per account; gaps were not checked.
* Observational data: associations, not causes. Check calibration over time before any deployment.